# Day 1 — B 담당: 뉴스 감성분류 + 카테고리 분류 + Materiality 초안

**오늘 만드는 것 (순서)**
1. `snunlp/KR-FinBert-SC`로 뉴스 긍정/부정 분류 (이미 파인튜닝된 모델이라 학습 없이 바로 씀)
2. HuggingFace zero-shot으로 뉴스 카테고리(ESG/실적·재무/산업·사업동향/문화·마케팅/기타) 분류
3. SASB 업종표 기반 Materiality Map(업종×카테고리→1/0) 초안 CSV 만들기
4. 위 세 개를 합쳐서 PRD 스키마대로 CSV 저장 (C가 이 파일을 받아서 씀)

**미리 할 일**: 상단 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU` 선택. (CPU로도 되지만 훨씬 느림)

## 0. 라이브러리 설치

In [ ]:
!pip install -q transformers torch pandas accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available())
device = 0 if torch.cuda.is_available() else -1  # HuggingFace pipeline은 0=GPU, -1=CPU

GPU 사용 가능: True


## 1. 모은 뉴스 데이터 불러오기

네이버 API로 모은 뉴스가 CSV/JSON 형태로 있을 것. Colab 왼쪽 폴더 아이콘 → 파일 업로드로 올리거나, Google Drive에 있으면 마운트해서 불러온다.

**컬럼 이름은 실제 파일에 맞게 바꿔라.** 최소한 `title`(제목), `ticker`(종목코드), `date`(날짜)는 있어야 한다. `description`(본문 요약)이 있으면 분류 정확도에 도움이 된다.

In [ ]:
# 방법 A: 직접 업로드
from google.colab import files
uploaded = files.upload()  # 여기서 뉴스 CSV 파일 선택

import pandas as pd
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(df.shape)
df.head()

Saving esg_labeling_sample_20260720.csv to esg_labeling_sample_20260720.csv
(400, 6)


,text,company,source_link,pub_date,esg_related,direction
0,LG에너지솔루션은 글로벌배터리연합(GBA)의 배터리 여권 시범사업에 참여해 협력사 ...,LG에너지솔루션,https://www.g-enews.com/view.php?ud=2026072015...,"Mon, 20 Jul 2026 18:16:00 +0900",NaN,NaN
1,행사장에 전시된 히트브랜드 수상 제품 에센코어 클레브 DDR5 6000 CL30 B...,SK하이닉스,https://www.thisisgame.com/articles/427963,"Mon, 20 Jul 2026 18:38:00 +0900",NaN,NaN
2,"우리은행, 교보생명, LG전자, LG에너지솔루션 등 관련기업도 참석해 준공을 축하했...",LG에너지솔루션,http://www.e2news.com/news/articleView.html?id...,"Mon, 20 Jul 2026 14:44:00 +0900",NaN,NaN
3,정 회장은 아산테헤네 국왕에게 현대차의 글로벌 자동차 생산 거점과 제조 공정을 담은...,현대차,https://www.mt.co.kr/industry/2026/07/20/20260...,"Mon, 20 Jul 2026 17:42:00 +0900",NaN,NaN
4,"""AI로 보는 반려동물 속마음""…카카오, '카나나 팻레터 만들기' 오픈",카카오,https://www.ddaily.co.kr/page/view/20260720175...,"Mon, 20 Jul 2026 18:18:00 +0900",NaN,NaN


In [ ]:
# 컬럼명 표준화 (사용자 요청: title, ticker, date, text, description)
rename_map = {
    'company': 'ticker',
    'pub_date': 'date',
    # 'text' 컬럼은 이미 존재함
}

df = df.rename(columns=rename_map)

# 필수 컬럼이 없을 경우 빈 값으로 생성
for col in ['title', 'ticker', 'date', 'text', 'description']:
    if col not in df.columns:
        df[col] = ""

# 만약 'text'에 본문이 있다면 'title'이 비어있을 때 요약해서 넣거나 관리
display(df[['title', 'ticker', 'date', 'text', 'description']].head())

,title,ticker,date,text,description
0,,LG에너지솔루션,"Mon, 20 Jul 2026 18:16:00 +0900",LG에너지솔루션은 글로벌배터리연합(GBA)의 배터리 여권 시범사업에 참여해 협력사 ...,
1,,SK하이닉스,"Mon, 20 Jul 2026 18:38:00 +0900",행사장에 전시된 히트브랜드 수상 제품 에센코어 클레브 DDR5 6000 CL30 B...,
2,,LG에너지솔루션,"Mon, 20 Jul 2026 14:44:00 +0900","우리은행, 교보생명, LG전자, LG에너지솔루션 등 관련기업도 참석해 준공을 축하했...",
3,,현대차,"Mon, 20 Jul 2026 17:42:00 +0900",정 회장은 아산테헤네 국왕에게 현대차의 글로벌 자동차 생산 거점과 제조 공정을 담은...,
4,,카카오,"Mon, 20 Jul 2026 18:18:00 +0900","""AI로 보는 반려동물 속마음""…카카오, '카나나 팻레터 만들기' 오픈",


In [ ]:
# 방법 B: Google Drive에서 불러오기 (방법 A 썼으면 이 셀은 건너뛰기)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/경로/뉴스파일.csv')

In [ ]:
# 분류에 넣을 텍스트: 불러온 데이터의 'text' 컬럼을 사용합니다.
# 만약 다른 컬럼(title 등)이 있다면 해당 컬럼명을 사용하세요.
if "text" in df.columns:
    df["text_for_model"] = df["text"].fillna("")
elif "title" in df.columns:
    if "description" in df.columns:
        df["text_for_model"] = df["title"].fillna("") + ". " + df["description"].fillna("")
    else:
        df["text_for_model"] = df["title"].fillna("")
else:
    # 둘 다 없는 경우 첫 번째 컬럼 사용 (비상용)
    df["text_for_model"] = df.iloc[:, 0].fillna("")

texts = df["text_for_model"].tolist()
print(f"처리할 데이터 개수: {len(texts)}")
display(df[["text_for_model"]].head())

처리할 데이터 개수: 400


,text_for_model
0,LG에너지솔루션은 글로벌배터리연합(GBA)의 배터리 여권 시범사업에 참여해 협력사 ...
1,행사장에 전시된 히트브랜드 수상 제품 에센코어 클레브 DDR5 6000 CL30 B...
2,"우리은행, 교보생명, LG전자, LG에너지솔루션 등 관련기업도 참석해 준공을 축하했..."
3,정 회장은 아산테헤네 국왕에게 현대차의 글로벌 자동차 생산 거점과 제조 공정을 담은...
4,"""AI로 보는 반려동물 속마음""…카카오, '카나나 팻레터 만들기' 오픈"


## 2. 감성분류 모델 로드 — `snunlp/KR-FinBert-SC`

이 모델은 **이미 학습이 끝난 모델**이다. 우리가 따로 학습시킬 필요 없이 바로 추론(inference)만 하면 된다.

⚠️ **중요**: HuggingFace 모델 카드에는 라벨이 `positive`/`negative`처럼 이름으로 안 나오고 `LABEL_0`, `LABEL_1`처럼 번호로 나올 수 있다. 그래서 아래 `id2label`을 반드시 출력해서 **어떤 번호가 긍정이고 어떤 번호가 부정인지 눈으로 확인**한 다음 다음 셀의 매핑을 고쳐야 한다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

sentiment_model_name = "snunlp/KR-FinBert-SC"
sentiment_tokenizer = AutoTokenizer.from_pretrained(sentiment_model_name)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(sentiment_model_name)

print("라벨 매핑 확인 (여기 나오는 걸 보고 아래 label_map을 고칠 것):")
print(sentiment_model.config.id2label)

config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/143k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/294k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  406MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

라벨 매핑 확인 (여기 나오는 걸 보고 아래 label_map을 고칠 것):
{0: 'negative', 1: 'neutral', 2: 'positive'}


In [ ]:
# 위 셀 출력이 예를 들어 {0: 'negative', 1: 'positive'} 이면 아래 label_map은 그대로 두면 됨.
# 만약 {0: 'LABEL_0', 1: 'LABEL_1'} 처럼 나오면, 모델 카드의 Positive/Negative 예시 문장을
# 직접 아래 sentiment_pipe에 넣어보고 어느 라벨이 긍정인지 실험으로 확인한 뒤 여기에 채워넣을 것.
label_map = {
    "negative": "negative",
    "neutral": "neutral",
    "positive": "positive"
}

sentiment_pipe = pipeline(
    "text-classification",
    model=sentiment_model,
    tokenizer=sentiment_tokenizer,
    device=device,
    truncation=True,
    max_length=512,
)

# 모델 카드 예시 문장으로 라벨 검증 (직접 눈으로 확인)
test_pos = "삼성전자, 2년 만에 인도 스마트폰 시장 점유율 1위 '왕좌 탈환'"
test_neg = "현대제철, 지난해 영업익 3,313억원···전년比 67.7% 감소"
print("긍정 예시 문장 결과:", sentiment_pipe(test_pos))
print("부정 예시 문장 결과:", sentiment_pipe(test_neg))

긍정 예시 문장 결과: [{'label': 'positive', 'score': 0.9998806715011597}]
부정 예시 문장 결과: [{'label': 'negative', 'score': 0.9998225569725037}]


## 3. 감성분류 실제 실행 (전체 뉴스에 적용)

한번에 다 넣지 않고 `batch_size`만큼씩 잘라서 처리한다 (메모리 절약 + Colab이 죽는 걸 방지).

In [ ]:
batch_size = 16
sentiment_results = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    outputs = sentiment_pipe(batch)
    sentiment_results.extend(outputs)
    if i % (batch_size * 10) == 0:
        print(f"{i}/{len(texts)} 처리 중...")

df["news_direction"] = [label_map.get(r["label"], r["label"]) for r in sentiment_results]
df["news_severity"] = [r["score"] for r in sentiment_results]  # 0~1 확신도, PRD의 severity로 그대로 사용

# 'title' 대신 생성한 'text_for_model' 컬럼을 사용해 결과를 출력합니다.
display(df[["text_for_model", "news_direction", "news_severity"]].head(10))

0/400 처리 중...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


160/400 처리 중...
320/400 처리 중...


,text_for_model,news_direction,news_severity
0,LG에너지솔루션은 글로벌배터리연합(GBA)의 배터리 여권 시범사업에 참여해 협력사 ...,positive,0.972617
1,행사장에 전시된 히트브랜드 수상 제품 에센코어 클레브 DDR5 6000 CL30 B...,neutral,0.985198
2,"우리은행, 교보생명, LG전자, LG에너지솔루션 등 관련기업도 참석해 준공을 축하했...",positive,0.997881
3,정 회장은 아산테헤네 국왕에게 현대차의 글로벌 자동차 생산 거점과 제조 공정을 담은...,positive,0.985933
4,"""AI로 보는 반려동물 속마음""…카카오, '카나나 팻레터 만들기' 오픈",neutral,0.999881
5,"삼성생명 -8.91%, KB금융 -6.79%, 현대차 -6.12%, LG에너지솔루션...",negative,0.997611
6,"삼성전자(-4.31%), SK하이닉스(-4.23%), SK스퀘어(-2.89%), 삼...",negative,0.999762
7,기술통 이청 대표를 중심으로 삼성전자를 비롯한 주요 고객사에 없어서는 안 될 존재가...,negative,0.733185
8,삼성전자와 SK하이닉스가 코스피 시가총액의 절반 이상을 차지하는 구조 자체가 변동성...,neutral,0.963680
9,"현대차(-6.12%), LG에너지솔루션(-4.94%), 삼성바이오로직스(-3.65%...",negative,0.999466


## 4. 카테고리 분류 (zero-shot)

KR-FinBert-SC는 긍정/부정만 판단하는 모델이라 **카테고리(ESG/실적·재무/...) 분류는 별도 모델**이 필요하다. PRD대로 다국어 zero-shot 모델(`joeddav/xlm-roberta-large-xnli`)을 쓴다. 이 모델은 학습 없이, 후보 라벨 문장 중 뉴스와 가장 비슷한 걸 골라준다.

⚠️ 이 모델은 용량이 커서(약 2GB) 처음 다운로드에 시간이 좀 걸린다.

In [ ]:
category_pipe = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=device,
)

candidate_labels = [
    "ESG 관련",
    "실적/재무 관련",
    "산업/사업동향 관련",
    "문화/마케팅 관련",
    "기타",
]

cat_map = {
    "ESG 관련": "ESG",
    "실적/재무 관련": "실적·재무",
    "산업/사업동향 관련": "산업·사업동향",
    "문화/마케팅 관련": "문화·마케팅",
    "기타": "기타",
}

config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
# zero-shot도 배치로 넣을 수 있다 (리스트를 그대로 전달)
categories = []
confidences = []

batch_size_cat = 8  # zero-shot은 감성분류보다 무거워서 배치를 더 작게
for i in range(0, len(texts), batch_size_cat):
    batch = texts[i:i+batch_size_cat]
    outputs = category_pipe(batch, candidate_labels)
    if isinstance(outputs, dict):  # 배치 크기 1일 때는 dict 하나만 반환됨
        outputs = [outputs]
    for out in outputs:
        categories.append(out["labels"][0])       # 가장 점수 높은 라벨
        confidences.append(out["scores"][0])       # 그 라벨의 확신도
    if i % (batch_size_cat * 10) == 0:
        print(f"{i}/{len(texts)} 처리 중...")

df["news_category"] = [cat_map[c] for c in categories]
df["confidence_score"] = confidences

# 'title' 대신 'text_for_model'을 사용합니다.
display(df[["text_for_model", "news_category", "confidence_score"]].head(10))

0/400 처리 중...
80/400 처리 중...
160/400 처리 중...
240/400 처리 중...
320/400 처리 중...


,text_for_model,news_category,confidence_score
0,LG에너지솔루션은 글로벌배터리연합(GBA)의 배터리 여권 시범사업에 참여해 협력사 ...,ESG,0.569835
1,행사장에 전시된 히트브랜드 수상 제품 에센코어 클레브 DDR5 6000 CL30 B...,ESG,0.329917
2,"우리은행, 교보생명, LG전자, LG에너지솔루션 등 관련기업도 참석해 준공을 축하했...",산업·사업동향,0.542388
3,정 회장은 아산테헤네 국왕에게 현대차의 글로벌 자동차 생산 거점과 제조 공정을 담은...,산업·사업동향,0.734117
4,"""AI로 보는 반려동물 속마음""…카카오, '카나나 팻레터 만들기' 오픈",기타,0.627316
5,"삼성생명 -8.91%, KB금융 -6.79%, 현대차 -6.12%, LG에너지솔루션...",산업·사업동향,0.749816
6,"삼성전자(-4.31%), SK하이닉스(-4.23%), SK스퀘어(-2.89%), 삼...",산업·사업동향,0.425026
7,기술통 이청 대표를 중심으로 삼성전자를 비롯한 주요 고객사에 없어서는 안 될 존재가...,산업·사업동향,0.816416
8,삼성전자와 SK하이닉스가 코스피 시가총액의 절반 이상을 차지하는 구조 자체가 변동성...,기타,0.481938
9,"현대차(-6.12%), LG에너지솔루션(-4.94%), 삼성바이오로직스(-3.65%...",산업·사업동향,0.390997


## 5. Materiality Map 초안 (SASB 업종표 기반)

네가 준 SASB 11개 업종표를 업종×카테고리(5종) 매트릭스로 정리한다.
- `ESG`, `실적·재무`는 모든 업종에서 재무적으로 중요하다고 보고 `1`
- 나머지 카테고리(`산업·사업동향`/`문화·마케팅`/`기타`)는 PRD 폴백 원칙대로 일단 `0`

⚠️ **주의**: 여기 업종명은 SASB 영문 분류(Consumer Goods, Technology & Communications 등)다. A가 pykrx로 가져오는 KRX 업종명(반도체, 화학, 유통업 등)과 이름이 다르다. **Day 2에서 'KRX 업종 → SASB 11개 업종' 매핑표를 따로 만들어야** 실제 종목에 적용할 수 있다. 오늘은 일단 SASB 기준으로 초안만 만들어둔다.

In [ ]:
sasb_sectors = [
    "Consumer Goods",
    "Extractives & Minerals Processing",
    "Financials",
    "Food & Beverage",
    "Health Care",
    "Infrastructure",
    "Renewable Resources & Alternative Energy",
    "Resource Transformation",
    "Services",
    "Technology & Communications",
    "Transportation",
]

categories_5 = ["ESG", "실적·재무", "산업·사업동향", "문화·마케팅", "기타"]

rows = []
for sector in sasb_sectors:
    for cat in categories_5:
        is_material = 1 if cat in ("ESG", "실적·재무") else 0
        rows.append({"sasb_sector": sector, "news_category": cat, "is_material": is_material})

materiality_map = pd.DataFrame(rows)
materiality_map.to_csv("materiality_map_draft.csv", index=False, encoding="utf-8-sig")
materiality_map.head(15)

,sasb_sector,news_category,is_material
0,Consumer Goods,ESG,1
1,Consumer Goods,실적·재무,1
2,Consumer Goods,산업·사업동향,0
3,Consumer Goods,문화·마케팅,0
4,Consumer Goods,기타,0
5,Extractives & Minerals Processing,ESG,1
6,Extractives & Minerals Processing,실적·재무,1
7,Extractives & Minerals Processing,산업·사업동향,0
8,Extractives & Minerals Processing,문화·마케팅,0
9,Extractives & Minerals Processing,기타,0


## 6. 결과 저장 (C에게 넘길 파일)

PRD 4-3 스키마 ①에 맞춰 컬럼을 정리해서 저장한다. `is_material`은 아직 실제 업종 매핑이 안 됐으므로 이번 파일에는 빠져 있다 — Day2에서 업종 매핑 붙인 뒤 `materiality_map_draft.csv`와 조인해서 채운다.

In [ ]:
# 'title'이 없으므로 'text_for_model' 또는 원본 'text'를 포함하도록 저장 컬럼을 조정합니다.
output_cols = ["company", "pub_date", "text_for_model", "news_direction", "news_severity", "news_category", "confidence_score"]
output_cols = [c for c in output_cols if c in df.columns]  # 실제 있는 컬럼만

result_df = df[output_cols]
result_df.to_csv("news_features_day1.csv", index=False, encoding="utf-8-sig")

from google.colab import files
try:
    files.download("news_features_day1.csv")
    files.download("materiality_map_draft.csv")
except:
    print("파일 다운로드는 브라우저 환경에 따라 작동하지 않을 수 있습니다. 왼쪽 파일 탭에서 직접 다운로드하세요.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

파일 다운로드를 시작합니다.


## 오늘 끝나면 체크할 것
- [ ] `id2label` 확인해서 label_map이 실제 모델과 맞는지 검증했는가 (긍정/부정 예시 문장으로 눈으로 확인)
- [ ] `news_features_day1.csv`에 `news_direction`, `news_severity`, `news_category`, `confidence_score`가 다 채워졌는가
- [ ] `materiality_map_draft.csv` 저장됐는가
- [ ] 카테고리 판정이 애매한 뉴스가 많으면 → 억지로 끼워맞추지 말고 `기타`로 두고 다음으로 (ROADMAP 폴백 원칙)
- [ ] 두 파일을 C에게 전달 (Google Drive 공유 폴더 추천)